This notebook is just a showcase of how we did the sampling. The CLI command `python -m mastodon_search choose-instances` does the same, but additionally probes the timelines API of all sampled instances, removes the ones that require a token or don't answer after a timeout. Because this takes a good amount of time, it is not part of this notebook.

In [ ]:
from pathlib import Path
from scipy.stats import lognorm
from numpy import geomspace
from pandas import concat
from seaborn import FacetGrid, histplot, set_theme

from mastodon_search.instance_data.analyze import Analyzer

In [ ]:
INSTANCE_DATA_PATH = Path('../data/instance_data.jsonl')
COLUMNS = {
    'total_users': "total users", 
    'monthly_users': "monthly active users", 
    'total_statuses': "total posts",
    'mean_weekly_statuses': "avg. weekly statuses",
    'mean_weekly_logins': "avg. weekly logins",
    'mean_weekly_registrations': "avg. weekly registrations"
}

In [ ]:
with open(INSTANCE_DATA_PATH, 'r') as file:
    an = Analyzer(file)

In [ ]:
cols_prob_measures = {
    col: lognorm
    # col: loguniform
    for col in COLUMNS.keys()
}

In [ ]:
df = an.df
df

In [ ]:
# Estimate probability distributions over activity columns
distributions = {
    col: dist(dist.fit(df[col]))
    for col, dist in cols_prob_measures.items()
}
distributions

In [ ]:
df_plot = concat([
    df.reset_index()[["instance", col]].rename(columns={col: "value"}).assign(statistic=name, col=col)
    for col, name in COLUMNS.items()
])
df_plot

In [ ]:
SAMPLE_INSTANCE_PATH = Path('../data/instances.txt')
with SAMPLE_INSTANCE_PATH.open("rt") as file:
    instances_sample = [line.strip() for line in file]
instances_sample[:10] 

In [ ]:
df_sample = df_plot[df_plot["instance"].isin(instances_sample)]
df_sample

In [ ]:
df_plot_all = concat([
    df_plot.assign(sampled=False),
    df_sample.assign(sampled=True),
])

In [ ]:
set_theme(style="white", context="paper", palette="colorblind")
df_this_plot = df_plot_all
# df_this_plot = df_plot_all[df_plot_all["statistic"].isin(["total users",  "avg. weekly logins"])]
g = FacetGrid(
    data=df_this_plot,
    col="statistic", 
    row="sampled",
    sharex="col",
    sharey="row",
)
g.map_dataframe(
    histplot,
    "value",
    hue="sampled",
    bins=10, 
    log_scale=(True, True),
    fill=True,
)
g.set(ylabel="Instances")
g.set_titles(template="")
for (sampled, name), ax in g.axes_dict.items():
    sampled: bool
    name: str
    ax2 = ax.twinx()
    ax.set_yscale("log")
    if ax.get_ylabel() == "Instances" and sampled:
        ax.set_ylabel("Sampled instances")
    if not sampled:
        ax2.set_yscale("log")
        col = next(c for c, n in COLUMNS.items() if n == name)
        dist = distributions[col]
        xlim = ax.get_xlim()
        min_x, max_x = xlim
        xs = geomspace(
            start=min_x,
            stop=max_x, 
            num=1000,
        )
        ys = [dist.pdf(i)[0] for i in xs]
        ax2.plot(xs, ys, color="red")
        ax.set_xlim(xlim)
    if sampled:
        ax2.set_yticks([])
        ax2.set_yticklabels([])
        ax.set_xlabel(name[:1].capitalize() + name[1:])
g.add_legend()
g.savefig("../data/histograms-instance-statistics-alt1.pdf")
g.figure.tight_layout()
g

In [ ]:
set_theme(style="white", context="paper", palette="colorblind")
df_this_plot = df_plot_all
# df_this_plot = df_plot_all[df_plot_all["statistic"].isin(["total users",  "avg. weekly logins"])]
g = FacetGrid(
    data=df_this_plot,
    col="statistic", 
    # row="sampled",
    sharex="col",
    sharey="row",
)
g.map_dataframe(
    histplot,
    "value",
    hue="sampled",
    bins=10, 
    log_scale=(True, True),
    fill=True,
)
g.set(ylabel="Instances")
g.set_titles(template="")
for name, ax in g.axes_dict.items():
    name: str
    ax2 = ax.twinx()
    ax.set_yscale("log")
    ax2.set_yscale("log")
    col = next(c for c, n in COLUMNS.items() if n == name)
    dist = distributions[col]
    xlim = ax.get_xlim()
    min_x, max_x = xlim
    xs = geomspace(
        start=min_x,
        stop=max_x, 
        num=1000,
    )
    ys = [dist.pdf(i)[0] for i in xs]
    ax2.plot(xs, ys, color="red")
    ax.set_xlim(xlim)
    ax.set_xlabel(name[:1].capitalize() + name[1:])
g.add_legend()
g.savefig("../data/histograms-instance-statistics-alt2.pdf")
g.figure.tight_layout()
g